In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline", force=True)
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display

PROJECT = Path(r"C:\Users\10856\Desktop\GithubProject\PlanRegion")
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
import planning_domain_demo as demo

# 重跑初始化块时，先释放上一次实验的 Gurobi 模型和图形资源。
for old_name in ("dual", "master"):
    old_model = globals().get(old_name)
    if old_model is not None:
        try: old_model.dispose()
        except Exception: pass
old_model = globals().get("model")
if old_model is not None:
    try: old_model.subproblem.dispose()
    except Exception: pass
plt.close("all")

plt.rcParams.update({"font.sans-serif": ["Microsoft YaHei", "SimHei", "DejaVu Sans"], "axes.unicode_minus": False})

TARGET_LAMBDA = 1.3  # 调参入口：固定负荷倍率，最小化建设费用
config = demo.DemoConfig()
model = demo.build_model(config)
master, choices, alpha = demo.build_master(model, mode="min_cost", query=TARGET_LAMBDA)
sp = model.subproblem

# 对偶模型只创建一次；第二块每轮仅更新目标，避免反复申请 Gurobi 资源。
dual = gp.Model("explicit_SP_dual")
pi_var = dual.addMVar(model.W.shape[0], lb=0, name="pi")
dual.addConstr(model.W.T @ pi_var == 0, name="stationarity")
dual.addConstr(model.scales @ pi_var <= 1, name="normalization")
SOLVE_TIME_LIMIT = 30.0
for solver in (master, sp, dual):
    solver.Params.OutputFlag, solver.Params.Threads = 0, 1
    solver.Params.TimeLimit = SOLVE_TIME_LIMIT
x_labels = [f"x[{e.name},{k.name}]" for e in model.corridors for k in model.lines]

cuts, history, finished = [], [], False
LB, UB = 0.0, np.inf
FEAS_TOL, CHECK_TOL = config.feasibility_tolerance, 1e-7

print(f"固定 λ={TARGET_LAMBDA}；总负荷={sum(config.loads_kw)*TARGET_LAMBDA:g} kW")
print(f"MP1：{model.nx} 个二元变量；SP：{model.W.shape[1]} 个潮流/电压变量 + eta；{model.W.shape[0]} 条电气约束")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2685996
Academic license 2685996 - for non-commercial use only - registered to 20___@mail.scut.edu.cn
固定 λ=1.3；总负荷=78 kW
MP1：15 个二元变量；SP：8 个潮流/电压变量 + eta；53 条电气约束


In [2]:
# 2A：求解 MP1
if finished: raise RuntimeError("实验已结束；请重跑第一块开始新实验。")
if not history: detail_history = {}
iteration, cuts_before = len(history) + 1, len(cuts)
round_key = (id(master), len(history))  # 防止同一轮重复加割、重复记账
previous = detail_history.get(iteration - 1)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda z: f"{z:.10g}")

master.optimize()
if master.Status == GRB.INFEASIBLE:
    finished = True
    raise RuntimeError("MP1 已证明不可行；本轮没有 SP，停止运行后续步骤。")
if master.Status != GRB.OPTIMAL: raise RuntimeError(f"MP1 status={master.Status}")

x_bar = np.rint([v.X for v in choices.values()])
lambda_bar, cost = alpha.X, float(model.cost @ x_bar)
LB = max(LB, master.ObjBound * 1000)
selected = [name for name, value in zip(x_labels, x_bar) if value > 0.5]

x_table = pd.DataFrame({"本轮 x": x_bar.astype(int), "费用系数/元": model.cost, "费用贡献/元": model.cost*x_bar}, index=x_labels)
if previous is not None:
    x_table["上一轮 x"] = previous["x"]
    x_table["变化 Δx"] = x_bar - previous["x"]

print(f"第 {iteration} 轮：λ={lambda_bar:g}，费用={cost:,.0f} 元，下界={LB:,.0f} 元")
display(x_table)
if cuts:
    display(pd.DataFrame({"已有割": range(1, len(cuts)+1), "当前方案处左端": [c.constant+c.x_coeff@x_bar+c.lambda_coeff*lambda_bar for c in cuts]}))

第 1 轮：λ=1.3，费用=0 元，下界=0 元


,本轮 x,费用系数/元,费用贡献/元
"x[01,L]",1,0,0
"x[01,M]",0,9460,-0
"x[01,H]",0,14300,-0
"x[12,L]",1,0,0
"x[12,M]",0,7740,-0
"x[12,H]",0,11700,-0
"x[13,L]",1,0,0
"x[13,M]",0,10320,-0
"x[13,H]",0,15600,-0
"x[02,L]",0,21900,-0


In [3]:
# 2B：读取系数，计算 b = h + T*x_bar + d*lambda_bar
assert round_key == (id(master), len(history)), "请从 2A 开始新一轮。"
rows = model.electrical_constraints
row_names = [c.ConstrName for c in rows]
w_names = [f"P[{e.name}]" for e in model.corridors] + ["v[1]", "v[2]", "v[3]"]
w_vars = [sp.getVarByName(f"P[{i}]") for i in range(len(model.corridors))] + [sp.getVarByName(f"v[{i}]") for i in range(1, 4)]
eta_var = sp.getVarByName("eta")
W, h, T, d, r = model.W, model.h, model.T, model.d, model.scales

# 从真实 LP 的左端重新读取 W 和 r，核对矩阵与求解器中的约束
W_read = np.array([[sp.getCoeff(c, v) for v in w_vars] for c in rows])
r_read = -np.array([sp.getCoeff(c, eta_var) for c in rows])
assert np.allclose(W, W_read) and np.allclose(r, r_read)

W_table = pd.DataFrame(W, index=row_names, columns=w_names)
T_table = pd.DataFrame(T, index=row_names, columns=x_labels)
Tx_terms = T * x_bar[None, :]
rhs = h + Tx_terms.sum(axis=1) + d*lambda_bar
rhs_table = pd.DataFrame({"h": h, "T*x": Tx_terms.sum(axis=1), "d": d, "lambda": lambda_bar, "d*lambda": d*lambda_bar, "b": rhs, "r": r}, index=row_names)

# None 显示全部53行；也可改成 ["capacity_01_1", "drop_upper_01_H"] 等行名列表
WATCH = None
show_rows = row_names if WATCH is None else WATCH

print("W：每条约束中，潮流/平方电压的系数")
display(W_table.loc[show_rows])
print("T：每条约束中，建设变量移到右端后的系数")
display(T_table.loc[show_rows])
print("T*x 的逐项乘积：只显示 x=1 的列，其余列乘积为0")
display(pd.DataFrame(Tx_terms, index=row_names, columns=x_labels).loc[show_rows, x_bar > 0.5])
print("右端计算：b = h + T*x + d*lambda；r 是 eta 的行尺度")
display(rhs_table.loc[show_rows])

# 从物理参数重新计算 drop_upper_01_H
ei = next(i for i, e in enumerate(model.corridors) if e.name == "01")
ki = next(i for i, k in enumerate(model.lines) if k.name == "H")
edge, line = model.corridors[ei], model.lines[ki]
q = np.tan(np.arccos(model.config.power_factor))
a = 2*(line.r_ohm_km+q*line.x_ohm_km)*(edge.length_m/1000)/(1000*model.config.voltage_kv**2)
M = 1-model.config.voltage_min_pu**2 + a*max(k.capacity_kw for k in model.lines)
i = row_names.index("drop_upper_01_H")

print(f"01:H 的压降系数 a={a:.12g}，大 M={M:.12g}")
print(f"{a:.12g}*P[01] + v[1] <= {1+M:.12g} - {M:.12g}*x[01,H] + {r[i]:.12g}*eta")
assert np.isclose(W[i, ei], a) and np.isclose(h[i], 1+M)
assert np.isclose(T[i, ei*len(model.lines)+ki], -M)

W：每条约束中，潮流/平方电压的系数


,P[01],P[12],P[13],P[02],P[23],v[1],v[2],v[3]
capacity_01_1,1,0,0,0,0,0,0,0
capacity_01_-1,-1,0,0,0,0,0,0,0
capacity_12_1,0,1,0,0,0,0,0,0
capacity_12_-1,0,-1,0,0,0,0,0,0
capacity_13_1,0,0,1,0,0,0,0,0
capacity_13_-1,0,0,-1,0,0,0,0,0
capacity_02_1,0,0,0,1,0,0,0,0
capacity_02_-1,0,0,0,-1,0,0,0,0
capacity_23_1,0,0,0,0,1,0,0,0
capacity_23_-1,0,0,0,0,-1,0,0,0


T：每条约束中，建设变量移到右端后的系数


,"x[01,L]","x[01,M]","x[01,H]","x[12,L]","x[12,M]","x[12,H]","x[13,L]","x[13,M]","x[13,H]","x[02,L]","x[02,M]","x[02,H]","x[23,L]","x[23,M]","x[23,H]"
capacity_01_1,35,65,100,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0
capacity_01_-1,35,65,100,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0
capacity_12_1,-0,-0,-0,35,65,100,-0,-0,-0,-0,-0,-0,-0,-0,-0
capacity_12_-1,-0,-0,-0,35,65,100,-0,-0,-0,-0,-0,-0,-0,-0,-0
capacity_13_1,-0,-0,-0,-0,-0,-0,35,65,100,-0,-0,-0,-0,-0,-0
capacity_13_-1,-0,-0,-0,-0,-0,-0,35,65,100,-0,-0,-0,-0,-0,-0
capacity_02_1,-0,-0,-0,-0,-0,-0,-0,-0,-0,35,65,100,-0,-0,-0
capacity_02_-1,-0,-0,-0,-0,-0,-0,-0,-0,-0,35,65,100,-0,-0,-0
capacity_23_1,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,35,65,100
capacity_23_-1,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,-0,35,65,100


T*x 的逐项乘积：只显示 x=1 的列，其余列乘积为0


,"x[01,L]","x[12,L]","x[13,L]"
capacity_01_1,35,-0,-0
capacity_01_-1,35,-0,-0
capacity_12_1,-0,35,-0
capacity_12_-1,-0,35,-0
capacity_13_1,-0,-0,35
capacity_13_-1,-0,-0,35
capacity_02_1,-0,-0,-0
capacity_02_-1,-0,-0,-0
capacity_23_1,-0,-0,-0
capacity_23_-1,-0,-0,-0


右端计算：b = h + T*x + d*lambda；r 是 eta 的行尺度


,h,T*x,d,lambda,d*lambda,b,r
capacity_01_1,0,35,-0,1.3,-0,35,60
capacity_01_-1,0,35,-0,1.3,-0,35,60
capacity_12_1,0,35,-0,1.3,-0,35,60
capacity_12_-1,0,35,-0,1.3,-0,35,60
capacity_13_1,0,35,-0,1.3,-0,35,60
capacity_13_-1,0,35,-0,1.3,-0,35,60
capacity_02_1,0,0,-0,1.3,-0,0,60
capacity_02_-1,0,0,-0,1.3,-0,0,60
capacity_23_1,0,0,-0,1.3,-0,0,60
capacity_23_-1,0,0,-0,1.3,-0,0,60


01:H 的压降系数 a=0.000952310503139，大 M=0.230331050314
0.000952310503139*P[01] + v[1] <= 1.23033105031 - 0.230331050314*x[01,H] + 0.1351*eta


In [4]:
# 2C：求解原始 SP
assert round_key == (id(master), len(history)), "请从 2A 开始新一轮。"
sp.setAttr("RHS", rows, rhs.tolist())
sp.optimize()
if sp.Status != GRB.OPTIMAL: raise RuntimeError(f"SP status={sp.Status}")
w_star, eta_star = np.array([v.X for v in w_vars]), sp.ObjVal

w_table = pd.DataFrame({"本轮 w*": w_star, "单位": ["kW"]*len(model.corridors)+["p.u.²"]*3}, index=w_names)
if previous is not None:
    w_table["上一轮 w*"] = previous["w"]
    w_table["变化 Δw"] = w_star - previous["w"]

print(f"SP 最优值 eta*={eta_star:.12g}；eta>0 时，以下潮流/电压是放松解。")
display(w_table)

Ww = W @ w_star
slack = rhs + r*eta_star - Ww
row_check = pd.DataFrame({"W*w": Ww, "b": rhs, "原约束残差 Ww-b": Ww-rhs, "允许放松 r*eta": r*eta_star, "LP余量 b+r*eta-Ww": slack, "归一化原违反量": np.maximum(Ww-rhs, 0)/r}, index=row_names)
display(row_check.loc[show_rows])
assert slack.min() >= -CHECK_TOL
print("原约束残差>0：违反原电气约束；LP余量≈0：放松后的约束取等号。")

SP 最优值 eta*=0.143333333333；eta>0 时，以下潮流/电压是放松解。


,本轮 w*,单位
P[01],43.6,kW
P[12],17.4,kW
P[13],2.3,kW
P[02],8.6,kW
P[23],8.6,kW
v[1],0.8783265954,p.u.²
v[2],0.8516389901,p.u.²
v[3],0.8895744951,p.u.²


,W*w,b,原约束残差 Ww-b,允许放松 r*eta,LP余量 b+r*eta-Ww,归一化原违反量
capacity_01_1,43.6,35,8.6,8.6,0,0.1433333333
capacity_01_-1,-43.6,35,-78.6,8.6,87.2,0
capacity_12_1,17.4,35,-17.6,8.6,26.2,0
capacity_12_-1,-17.4,35,-52.4,8.6,61,0
capacity_13_1,2.3,35,-32.7,8.6,41.3,0
capacity_13_-1,-2.3,35,-37.3,8.6,45.9,0
capacity_02_1,8.6,0,8.6,8.6,0,0.1433333333
capacity_02_-1,-8.6,0,-8.6,8.6,17.2,0
capacity_23_1,8.6,0,8.6,8.6,0,0.1433333333
capacity_23_-1,-8.6,0,-8.6,8.6,17.2,0


原约束残差>0：违反原电气约束；LP余量≈0：放松后的约束取等号。


In [5]:
# 2D：读取 pi，并独立求解对偶 LP
assert round_key == (id(master), len(history)), "请从 2A 开始新一轮。"
gurobi_pi = np.array(sp.getAttr("Pi", rows))
pi = -gurobi_pi

dual = gp.Model("explicit_dual")
dual.Params.OutputFlag, dual.Params.Threads = 0, 1
dual.Params.FeasibilityTol, dual.Params.OptimalityTol = 1e-9, 1e-9
pi_var = dual.addMVar(len(rows), lb=0, name="pi")
dual.addConstr(W.T @ pi_var == 0)
dual.addConstr(r @ pi_var <= 1)
dual.setObjective(-rhs @ pi_var, GRB.MAXIMIZE)
dual.optimize()
if dual.Status != GRB.OPTIMAL: raise RuntimeError(f"dual status={dual.Status}")
pi_explicit, dual_value = pi_var.X.copy(), dual.ObjVal

pi_table = pd.DataFrame({"Gurobi Pi": gurobi_pi, "用于割的 pi=-Pi": pi, "独立对偶解": pi_explicit, "b_i": rhs, "对偶目标贡献 -pi_i*b_i": -pi*rhs, "LP余量": slack, "互补乘积 pi_i*余量": pi*slack}, index=row_names)
if previous is not None: pi_table["pi 的轮间变化"] = pi - previous["pi"]

display(pi_table.loc[show_rows])
display(pd.DataFrame({"Wᵀ*pi（应为0）": W.T @ pi, "独立解 Wᵀ*pi": W.T @ pi_explicit}, index=w_names))

dual_error = max(abs(eta_star-dual_value), abs(eta_star+pi@rhs))
print(f"原始目标={eta_star:.12g}；-piᵀb={-pi@rhs:.12g}；独立对偶目标={dual_value:.12g}")
print(f"rᵀpi={r@pi:.12g}（应<=1）；eta*(1-rᵀpi)={eta_star*(1-r@pi):.3g}（应≈0）")
assert pi.min() >= -CHECK_TOL and np.max(np.abs(W.T @ pi)) <= CHECK_TOL
assert r @ pi <= 1+CHECK_TOL and dual_error <= CHECK_TOL
assert np.max(np.abs(pi*slack)) <= CHECK_TOL

,Gurobi Pi,用于割的 pi=-Pi,独立对偶解,b_i,对偶目标贡献 -pi_i*b_i,LP余量,互补乘积 pi_i*余量
capacity_01_1,-0.003333333333,0.003333333333,0.003333333333,35,-0.1166666667,0,0
capacity_01_-1,0,-0,0,35,0,87.2,-0
capacity_12_1,0,-0,0,35,0,26.2,-0
capacity_12_-1,0,-0,0,35,0,61,-0
capacity_13_1,0,-0,0,35,0,41.3,-0
capacity_13_-1,0,-0,0,35,0,45.9,-0
capacity_02_1,-0.003333333333,0.003333333333,0.003333333333,0,-0,0,0
capacity_02_-1,0,-0,0,0,0,17.2,-0
capacity_23_1,0,-0,0,0,0,0,-0
capacity_23_-1,0,-0,0,0,0,17.2,-0


,Wᵀ*pi（应为0）,独立解 Wᵀ*pi
P[01],0,0
P[12],0,0
P[13],0,8.67361738e-19
P[02],0,0
P[23],0,8.67361738e-19
v[1],0,0
v[2],0,0
v[3],0,0


原始目标=0.143333333333；-piᵀb=0.143333333333；独立对偶目标=0.143333333333
rᵀpi=1（应<=1）；eta*(1-rᵀpi)=0（应≈0）


In [6]:
# 2E：每条约束对割系数的贡献，以及当前方案代入后的数值
assert round_key == (id(master), len(history)), "请从 2A 开始新一轮。"
active = np.flatnonzero(np.abs(pi) > 1e-10)

# 查看加权相加后，w 的各个系数如何抵消
cancel_table = pd.DataFrame(pi[:, None]*W, index=row_names, columns=w_names)
cancel_show = cancel_table.iloc[active].copy()
cancel_show.loc["全部53行合计"] = cancel_table.sum(axis=0)
print("逐行乘 pi 后，潮流/电压系数如何相互抵消：")
display(cancel_show)

# 每行贡献 pi_i*h_i、pi_i*T_ij、pi_i*d_i
coefficient_names = ["常数"] + x_labels + ["lambda"]
contribution = pd.DataFrame(pi[:, None]*np.column_stack([h, T, d]), index=row_names, columns=coefficient_names)
visible_cols = contribution.abs().max(axis=0) > 1e-10
contribution_show = contribution.iloc[active].loc[:, visible_cols].copy()
contribution_show.loc["全部53行合计"] = contribution.sum(axis=0).loc[visible_cols]
print("每条电气约束对割系数的贡献；最后一行就是最终系数：")
display(contribution_show)

beta0, beta_x, beta_lambda = float(pi @ h), pi @ T, float(pi @ d)
cut = demo.DualCut(pi=pi.copy(), constant=beta0, x_coeff=beta_x, lambda_coeff=beta_lambda, source_violation=eta_star)
beta = np.r_[beta0, beta_x, beta_lambda]
assert np.allclose(contribution.sum(axis=0).to_numpy(), beta)

# 将当前方案逐项代入割
point = np.r_[1.0, x_bar, lambda_bar]
evaluation = pd.DataFrame({"割系数 beta": beta, "当前变量值": point, "相乘结果": beta*point}, index=coefficient_names)
evaluation_show = evaluation.loc[np.abs(beta) > 1e-10].copy()
evaluation_show.loc["全部项合计", "相乘结果"] = float(beta @ point)
display(evaluation_show)

margin = float(beta @ point)
terms = [f"{a:+.10g}*{name}" for name, a in zip(x_labels, beta_x) if abs(a) > 1e-10]
print("割：", f"{beta0:.10g} " + " ".join(terms) + f" {beta_lambda:+.10g}*lambda >= 0")
print(f"当前方案处左端={margin:.12g}；-eta*={-eta_star:.12g}")
assert abs(margin+eta_star) <= CHECK_TOL

逐行乘 pi 后，潮流/电压系数如何相互抵消：


,P[01],P[12],P[13],P[02],P[23],v[1],v[2],v[3]
capacity_01_1,0.003333333333,0,0,0,0,0,0,0
capacity_02_1,0,0,0,0.003333333333,0,0,0,0
balance_1_-1,-0.003333333333,0.003333333333,0.003333333333,0,0,0,0,0
balance_2_-1,0,-0.003333333333,0,-0.003333333333,0.003333333333,0,0,0
balance_3_-1,0,0,-0.003333333333,0,-0.003333333333,0,0,0
全部53行合计,0,0,0,0,0,0,0,0


每条电气约束对割系数的贡献；最后一行就是最终系数：


,"x[01,L]","x[01,M]","x[01,H]","x[02,L]","x[02,M]","x[02,H]",lambda
capacity_01_1,0.1166666667,0.2166666667,0.3333333333,-0,-0,-0,-0
capacity_02_1,-0,-0,-0,0.1166666667,0.2166666667,0.3333333333,-0
balance_1_-1,-0,-0,-0,-0,-0,-0,-0.08333333333
balance_2_-1,-0,-0,-0,-0,-0,-0,-0.06666666667
balance_3_-1,-0,-0,-0,-0,-0,-0,-0.05
全部53行合计,0.1166666667,0.2166666667,0.3333333333,0.1166666667,0.2166666667,0.3333333333,-0.2


,割系数 beta,当前变量值,相乘结果
"x[01,L]",0.1166666667,1,0.1166666667
"x[01,M]",0.2166666667,-0,-0
"x[01,H]",0.3333333333,-0,-0
"x[02,L]",0.1166666667,-0,-0
"x[02,M]",0.2166666667,-0,-0
"x[02,H]",0.3333333333,-0,-0
lambda,-0.2,1.3,-0.26
全部项合计,NaN,NaN,-0.1433333333


割： 0 +0.1166666667*x[01,L] +0.2166666667*x[01,M] +0.3333333333*x[01,H] +0.1166666667*x[02,L] +0.2166666667*x[02,M] +0.3333333333*x[02,H] -0.2*lambda >= 0
当前方案处左端=-0.143333333333；-eta*=-0.143333333333


In [7]:
# 2F：加入割或宣布收敛；只在这里推进迭代状态
assert round_key == (id(master), len(history)), "本轮已记录；请从 2A 开始下一轮。"

if eta_star <= FEAS_TOL:
    UB, finished, action = min(UB, cost), True, "收敛"
    print(f"电气可行：LB={LB:,.0f} 元，UB={UB:,.0f} 元。")
else:
    assert margin < -FEAS_TOL
    cuts.append(cut)
    demo.add_dual_cut(master, choices, alpha, cut)
    master.update()
    action = "加割"
    scale = max(1.0, np.max(np.abs(beta_x)), abs(beta_lambda), abs(beta0))
    print(f"加入第 {len(cuts)} 条割；原函数将整条割除以 {scale:.10g}，只改善数值尺度。")
    print("本轮结束；回到 2A 求解下一轮 MP1。")

history.append({"iteration": iteration, "cuts_before": cuts_before, "cost": cost, "LB": LB, "UB": UB, "eta": eta_star, "dual": dual_value, "dual_error": dual_error, "cut_margin": margin, "action": action, "design": "; ".join(selected)})
detail_history[iteration] = {"x": x_bar.copy(), "w": w_star.copy(), "eta": eta_star, "pi": pi.copy(), "rhs": rhs.copy(), "rows": row_check.copy(), "contribution": contribution.copy(), "cut": cut}
display(pd.DataFrame(history)[["iteration", "cost", "LB", "UB", "eta", "cut_margin", "action"]])

加入第 1 条割；原函数将整条割除以 1，只改善数值尺度。
本轮结束；回到 2A 求解下一轮 MP1。


,iteration,cost,LB,UB,eta,cut_margin,action
0,1,0,0,inf,0.1433333333,-0.1433333333,加割
